## 순수함수
- 동일한 입력이 주어지면 항상 동일한 값을 반환하며, 함수 외부의 상태를 변경하거나 영향을 받지 않는(부수 효과가 없는) 함수
$f(x)=x^2+1$ 와 유사


In [ ]:
# 순수 함수 예시
def add(a, b):
    return a + b

In [ ]:
# 비순수 함수 예시 1: 외부 변수 참조 및 변경
counter = 0

def increment():
    global counter
    counter += 1
    return counter

# 비순수 함수 예시 2: 시간이나 랜덤 값 의존
import random

def get_random_number():
    return random.randint(1, 10)

## 람다 대수
$\lambda x.x+1$

In [2]:
double = lambda x: x * 2

print(double(5))

10


In [4]:
# 클로저
def outer(x):
    def inner(y):
        return x + y
    return inner

add5 = outer(5)
add10 = outer(10)

print(add5(3)) # 8
print(add10(3)) # 13

print(add5(7)) # 12

8
13
12


## 자유변수
- 함수내부에서 사용되나 해당 함수 안에서 정의되지 않은 함수
- 클로저에서 내부 함수가 참조하는 외부 함수의 변수

`Local(지역) -> Enclosing(감싸는 함수의 블럭) -> Global(모듈 전역) -> Built-in(내장)`

- 내부 함수에서 외부 함수의 변수를 읽는 것은 자동으로 자유 변수가 처리
- 하지만 변수가 재할당 되려면 반드시 `nonlocal` 키워드 사용해야함
- `nolocal` 전역 스코프가 아니라 `Enclosing` 스코프 변수가 수정할 때 사용

In [9]:
x = 'global' # (1) 전역 변수 x 선언

def outer_func(x):
    x = 'enclosing' # (2) 매개변수 x를 'enclosing'으로 재할당

    def inner_func(y):
        x = 'local' # (3) inner_func 내의 지역변수 x 선언
        return x

    print(x) # (4) 여기서 'enclosing'이 출력됨
    return inner_func # (5) inner_func 함수 객체를 반환

# 실행부
func = outer_func('test') # (6) outer_func 호출 -> 'enclosing' 출력됨
print(func(10)) # (7) inner_func 호출 -> 'local' 출력됨
print(x) # (8) 전역 변수 'global' 출력


print(closure_instance.__closure__) # (<cell at 0x000001A31A0A7F40: int object at 0x00007FFDDA54C598>,)

enclosing
local
global
(<cell at 0x000001A31A0A7F40: int object at 0x00007FFDDA54C598>,)


In [8]:
def make_colsure():
    count = 0
    def counter():
        nonlocal count
        count += 1
        return count
    return counter

In [10]:
def outer():
    x = 1
    def outer2():
        def inner():
            nonlocal  x
            x += 10
        inner()
        print(x)
    outer2()

outer()

11


## 데코레이터
- https://wikidocs.net/160127
- 어떤 함수가 있을 때 해당 함수를 직접 수정하지 않고 함수에 기능을 추가하고자 할 때 사용
- 본질적으로 다른 함수의 인자를 받음. 해당 함수의 동작에 추가된 새로운 함수를 반환하는 고차함수
- 즉, 클로저 기반으로 구현됨


In [16]:
# 실행시간 측정하는 기능...
import time

def heavy_add(a,b):
    time.sleep(1)
    return a+b

def heavy_multiply(a, b):
    time.sleep(1)
    return a * b

start = time.time()
result = heavy_add(1,2)
elapsed = time.time() - start

print(f'heavy_add는 {elapsed}초, 결과는 {result}')
# 그럼 5번해야한다면........

heavy_add는 1.000258445739746초, 결과는 3


In [15]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = heavy_add(*args, **kwargs)
        elapsed = time.time() - start

        print(f'{func.__name__}:{elapsed:.2f}초, 결과는{result}')
        return result
    return wrapper

timer_add = timer(heavy_add)
timer_add(5,5)


heavy_add:1.00초, 결과는10


10

## @ 데코레이터 문법쓰기

In [20]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        # 수정 전: result = heavy_add(*args, **kwargs)
        # 수정 후: 아래와 같이 func를 호출해야 합니다.
        result = func(*args, **kwargs)
        elapsed = time.time() - start

        print(f'{func.__name__}:{elapsed:.2f}초, 결과는{result}')
        return result
    return wrapper

@timer
def heavy_add(a,b):
    time.sleep(1)
    return a+b

@timer
def heavy_multiply(a, b):
    time.sleep(1)
    return a * b

heavy_add(5, 5)
heavy_multiply(5, 5)

heavy_add:1.00초, 결과는10
heavy_multiply:1.00초, 결과는25


25

In [26]:
# 인자를 받는 데코레이터
import functools

def limit_calls(max_calls):

    def decorator(func):
        call_count = 0
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            nonlocal  call_count

            if call_count >= max_calls:
                raise Exception('Max calls reached')
            call_count += 1
            print(f'Call count: {call_count}/{max_calls}호출시도')
            return func(*args, **kwargs)
        return wrapper
    return decorator


@limit_calls(3)
def heavy_add(a,b):
    time.sleep(1)
    return a+b

print(heavy_add(5, 5))
print(heavy_add(5, 5))
print(heavy_add(5, 5))
#heavy_add(5, 5) # 3번호출하기로 해서 그이상은 안됨


Call count: 1/3호출시도
10
Call count: 2/3호출시도
10
Call count: 3/3호출시도
10


## 제너레이터(Generator)
- 데이터를 한 번에 메모리에 올리지 않고, 필요할 때마다 하나씩 계산해서 반환(Yield)하는 함수 -> lazy방식
- 일반적인 함수는 return을 만나면 값을 반환하고 종료되지만, 제너레이터는 yield를 만나면 값을 반환하고 함수 실행을 일시 정지
- 이후 다시 호출되면 정지했던 지점부터 다시 실행

In [27]:
def countdown(n):
    print(f'Countdown: {n}')
    while n > 0:
        yield n  # 현재 n 값을 반환하고 여기서 실행을 멈춤
        n -= 1   # 다음 호출 시 이 지점부터 다시 시작

# 사용 예시
gen = countdown(5)

for num in gen:
    print(num)

Countdown: 5
5
4
3
2
1
